# Video Embedding Setup

This notebook prepares video embeddings for use in subsequent notebooks (02, 03, 04).

If you already generated embeddings in Workshop 1 (`1_basic`), they will be **automatically detected and reused**. New embeddings are generated with Marengo 3.0 only when no existing embeddings are found.

To test with your own videos, place `.mp4` files in the `videos/` folder.

## 0. Setup

In [ ]:
%pip install -r requirements.txt -Uq

In [ ]:
import json, time, os
import boto3
from botocore.config import Config

session = boto3.Session()
REGION = session.region_name or "us-east-1"
AWS_ACCOUNT_ID = session.client("sts").get_caller_identity()["Account"]
bedrock = session.client("bedrock-runtime", region_name=REGION, config=Config(signature_version="v4"))
s3 = session.client("s3", region_name=REGION)

print(f"✅ Region: {REGION}, Account: {AWS_ACCOUNT_ID}")

## 1. Workshop Resources

Enter the workshop resources provided by CloudFormation outputs. These values are saved to `config.json` and automatically loaded by subsequent notebooks.

In [ ]:
# ==============================
# TODO: Enter your workshop resources
# ==============================
S3_BUCKET = "<YOUR_S3_BUCKET_NAME>"                          # WorkshopS3BucketName
OPENSEARCH_ENDPOINT = "<YOUR_OPENSEARCH_ENDPOINT>"           # OpenSearchVectorDBEndpoint
S3_VECTOR_BUCKET_NAME = "<YOUR_S3_VECTOR_BUCKET_NAME>"       # WorkshopS3VectorBucketName

# Validate
if S3_BUCKET.startswith("<"):
    raise ValueError("Please enter S3_BUCKET (CloudFormation output: WorkshopS3BucketName)")
if OPENSEARCH_ENDPOINT.startswith("https://"):
    OPENSEARCH_ENDPOINT = OPENSEARCH_ENDPOINT.replace("https://", "")
if OPENSEARCH_ENDPOINT.startswith("<"): OPENSEARCH_ENDPOINT = ""
if S3_VECTOR_BUCKET_NAME.startswith("<"): S3_VECTOR_BUCKET_NAME = ""

VIDEO_S3_PREFIX = "vectordb-workshop/videos/"
EMBEDDINGS_S3_PREFIX = "vectordb-workshop/embeddings/"
MODEL_ID = "twelvelabs.marengo-embed-3-0-v1:0"

print(f"S3 Bucket: {S3_BUCKET}")
print(f"   OpenSearch: {OPENSEARCH_ENDPOINT or '(not set)'}")
print(f"   S3 Vector Bucket: {S3_VECTOR_BUCKET_NAME or '(not set)'}")

## 2. Detect Existing Embeddings

Search S3 for existing embedding files. If embeddings from Workshop 1 are found, they will be automatically reused.

In [ ]:
def find_existing_embeddings():
    """Search for existing embedding files across known S3 paths."""
    for prefix in [EMBEDDINGS_S3_PREFIX, "embeddings/videos/", "embeddings/"]:
        resp = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=prefix, MaxKeys=100)
        files = [o for o in resp.get("Contents", []) if o["Key"].endswith(".json")]
        if files:
            return prefix, files
    return None, []

def load_embedding_file(s3_key):
    """Load embedding file (handles Workshop 1 and 4 formats)."""
    data = json.loads(s3.get_object(Bucket=S3_BUCKET, Key=s3_key)["Body"].read().decode())
    if "vectors" in data: return data["vectors"]
    if "data" in data: return data["data"]
    vecs = []
    for rec in data.get("embeddings", [data]):
        for seg in rec.get("data", []):
            if "embedding" in seg: vecs.append(seg)
    return vecs

found_prefix, found_files = find_existing_embeddings()
EMBEDDINGS_EXIST = False
all_embeddings = {}

if found_files:
    print(f"Found {len(found_files)} existing embedding files: s3://{S3_BUCKET}/{found_prefix}")
    for f in found_files:
        vecs = load_embedding_file(f["Key"])
        if not vecs or "embedding" not in vecs[0]: continue
        basename = os.path.splitext(os.path.basename(f["Key"]))[0]
        vid = f["Key"].split("/")[-3] if basename == "output" else basename
        all_embeddings[vid] = {"video_id": vid, "vectors": vecs}
        print(f"   {vid}: {len(vecs)} vectors")
    if all_embeddings:
        EMBEDDINGS_EXIST = True
        EMBEDDINGS_S3_PREFIX = found_prefix
        print(f"\nReusing existing embeddings. Skip to Section 5.")

if not EMBEDDINGS_EXIST:
    print("No existing embeddings found. Upload videos in Section 3, then generate embeddings in Section 4.")

## 3. Upload Videos to S3

**Skip this section if existing embeddings were found.**

Place `.mp4` files in the `videos/` folder and they will be automatically uploaded to S3.

In [ ]:
if EMBEDDINGS_EXIST:
    print("Skipping video upload — using existing embeddings")
else:
    LOCAL_VIDEO_DIR = "videos"
    resp = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=VIDEO_S3_PREFIX, MaxKeys=100)
    existing = {os.path.basename(o["Key"]) for o in resp.get("Contents", []) if o["Key"].endswith(".mp4")}

    if os.path.isdir(LOCAL_VIDEO_DIR):
        for vf in sorted(os.listdir(LOCAL_VIDEO_DIR)):
            if not vf.endswith(".mp4"): continue
            if vf in existing:
                print(f"   Skipped: {vf} (already in S3)")
            else:
                print(f"   Uploading: {vf}...", end=" ")
                s3.upload_file(os.path.join(LOCAL_VIDEO_DIR, vf), S3_BUCKET, f"{VIDEO_S3_PREFIX}{vf}")
                print("Done")

    resp = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=VIDEO_S3_PREFIX, MaxKeys=100)
    video_keys = [o["Key"] for o in resp.get("Contents", []) if o["Key"].endswith(".mp4")]
    print(f"\n{'Found' if video_keys else 'No'} S3 videos: {len(video_keys)}")

## 4. Generate Embeddings

**Skip this section if existing embeddings were found.**

Marengo 3.0 splits a video into time-based segments and generates a 512-dimensional vector for each:
- **clip-level**: per-segment embeddings
- **asset-level**: whole-video summary embedding

In [ ]:
EMBEDDING_MODALITY = "visual"
POLL_INTERVAL = 15

In [ ]:
if EMBEDDINGS_EXIST:
    print("Skipping embedding generation — using existing embeddings")
else:
    def start_embedding_job(video_s3_key):
        video_id = os.path.splitext(os.path.basename(video_s3_key))[0]
        resp = bedrock.start_async_invoke(
            modelId=MODEL_ID,
            modelInput={"inputType": "video", "video": {
                "mediaSource": {"s3Location": {"uri": f"s3://{S3_BUCKET}/{video_s3_key}", "bucketOwner": AWS_ACCOUNT_ID}},
                "embeddingOption": [EMBEDDING_MODALITY], "embeddingScope": ["clip", "asset"]
            }},
            outputDataConfig={"s3OutputDataConfig": {"s3Uri": f"s3://{S3_BUCKET}/{EMBEDDINGS_S3_PREFIX}{video_id}/"}}
        )
        return resp["invocationArn"], video_id

    def wait_for_job(arn, timeout=600):
        start = time.time()
        while time.time() - start < timeout:
            job = bedrock.get_async_invoke(invocationArn=arn)
            status = job["status"]
            if status == "Completed": return job
            if status in ("Failed", "Expired"): return None
            print(f"  [{time.strftime('%H:%M:%S')}] {status}"); time.sleep(POLL_INTERVAL)
        return None

    def download_embeddings(video_id):
        prefix = f"{EMBEDDINGS_S3_PREFIX}{video_id}/"
        resp = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=prefix)
        results = []
        for obj in resp.get("Contents", []):
            if obj["Key"].endswith((".json", ".jsonl")):
                raw = s3.get_object(Bucket=S3_BUCKET, Key=obj["Key"])["Body"].read().decode()
                for line in (raw.strip().split("\n") if obj["Key"].endswith(".jsonl") else [raw]):
                    if line.strip(): results.append(json.loads(line))
        return results

    resp = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=VIDEO_S3_PREFIX)
    video_keys = [o["Key"] for o in resp.get("Contents", []) if o["Key"].endswith(".mp4")]

    jobs = []
    for vk in video_keys:
        print(f"  Starting: {os.path.basename(vk)}...", end=" ")
        try:
            arn, vid = start_embedding_job(vk); jobs.append({"arn": arn, "video_id": vid}); print("Done")
        except Exception as e: print(f"Error: {e}")

    for job in jobs:
        vid = job["video_id"]
        print(f"\nWaiting: {vid}...")
        if wait_for_job(job["arn"]):
            raw = download_embeddings(vid)
            vectors = [seg for rec in raw for seg in rec.get("data", []) if "embedding" in seg]
            all_embeddings[vid] = {"video_id": vid, "vectors": vectors}
            print(f"  {len(vectors)} vectors")

    for vid, data in all_embeddings.items():
        s3.put_object(Bucket=S3_BUCKET, Key=f"{EMBEDDINGS_S3_PREFIX}{vid}.json",
                      Body=json.dumps(data), ContentType="application/json")
    print(f"\nTotal: {sum(len(v['vectors']) for v in all_embeddings.values())} vectors generated")

## 5. Save Configuration

Verify the embedding structure and save `config.json`. Subsequent notebooks (02, 03, 04) will automatically load this file.

In [ ]:
for vid, data in all_embeddings.items():
    vectors = data["vectors"]
    asset = [v for v in vectors if v.get("embeddingScope") == "asset"]
    clips = [v for v in vectors if v.get("embeddingScope") == "clip"]
    print(f"{vid}: {len(vectors)} vectors (asset={len(asset)}, clip={len(clips)})")

first_vid = next(iter(all_embeddings.values()))
EMBEDDING_DIMENSION = len(first_vid["vectors"][0]["embedding"])

config = {
    "REGION": REGION, "S3_BUCKET": S3_BUCKET, "AWS_ACCOUNT_ID": AWS_ACCOUNT_ID,
    "EMBEDDINGS_S3_PREFIX": EMBEDDINGS_S3_PREFIX, "EMBEDDING_DIMENSION": EMBEDDING_DIMENSION,
    "VIDEO_IDS": list(all_embeddings.keys()),
    "OPENSEARCH_ENDPOINT": OPENSEARCH_ENDPOINT, "S3_VECTOR_BUCKET_NAME": S3_VECTOR_BUCKET_NAME,
    "VIDEO_S3_PREFIX": VIDEO_S3_PREFIX
}
with open("config.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"\nconfig.json saved (dimension={EMBEDDING_DIMENSION}, videos={len(all_embeddings)})")
print(f"   Next: Run 02_opensearch_serverless.ipynb or 03_s3_vectors.ipynb")